# sqd-go — Quickstart: derive state with a custom processor

This builds a real indexer: as LBTC `Transfer` events stream in, a **custom processor**
maintains a per-address position (total in / out / transfer count) in a ClickHouse table
you define. You will:

1. install Go, the `sqd-go` CLI, and ClickHouse,
2. scaffold a project (`config.yaml` + `custom_schema.go` + `custom_processor.go`),
3. index a range with `--state` (which compiles your processor in), and
4. query both the raw events **and** the derived per-address state.

Along the way we cover the two things that trip up newcomers: **how Go modules work here**
and **proto vs. `--no-proto` mode**.

## 1. Install Go

`sqd-go` is a Go program **and** a code generator: it generates Go for your indexer and
compiles it, so a working Go toolchain is required. This installs Go into `/root/.go`.

In [ ]:
# Install Go (~30s)
!wget -q -O - https://raw.githubusercontent.com/canha/golang-tools-install-script/master/goinstall.sh | bash >/dev/null 2>&1

In [ ]:
import os
# Make Go and any `go install`-ed binaries (like sqd-go) visible to every shell cell.
os.environ['GOROOT'] = '/root/.go'
os.environ['GOPATH'] = '/root/go'
os.environ['PATH']   = '/root/.go/bin:/root/go/bin:' + os.environ['PATH']
!go version

## 2. Install the sqd-go CLI

The install script runs `go install github.com/franz101/sqd-go@latest`, dropping the
`sqd-go` binary into `$(go env GOPATH)/bin` (already on `PATH` from the cell above).

In [ ]:
# Install the CLI (first run compiles dependencies, ~1-2 min)
!curl -sSL https://raw.githubusercontent.com/franz101/sqd-go/main/install.sh | bash
!sqd-go help | head -20

## 3. How Go modules work here

Your indexer is its own Go module. `sqd-go start <proj> --state` scaffolds a `go.mod`, generates the code, then builds and runs it. Two rules keep it working:

1. **One package per directory.** `custom_schema.go` and `custom_processor.go` must share the same `package`.
2. **Import the public facade, never `internal/`.** Use `github.com/franz101/sqd-go/sqd` and your own `"<module>/generated"`.

New to Go? See **Appendix — Go for beginners** at the bottom. Full detail: [`GO_MODULES.md`](../wiki/GO_MODULES.md).

## 4. Run ClickHouse

sqd-go indexes into [ClickHouse](https://clickhouse.com). In Colab we use the official
single binary with an **empty password** on the default ports (native `9000`, HTTP `8123`).

In [ ]:
# Download the ClickHouse single binary
!curl -s https://clickhouse.com/ | sh >/dev/null 2>&1

# Start the server in the background (empty password) and give it a moment to come up
import subprocess, time
subprocess.Popen(['./clickhouse', 'server'],
                 stdout=open('clickhouse-server.log','w'),
                 stderr=subprocess.STDOUT)
time.sleep(8)
!./clickhouse client --query "SELECT 'ClickHouse is up' AS status"

## 5. Scaffold the project

Same `config.yaml` as the minimal demo — chain, range, contract, and the `Transfer` event.

In [ ]:
!mkdir -p lbtcdemo

In [ ]:
%%writefile lbtcdemo/config.yaml
name: lbtcdemo
chains:
  - id: 1
    start_block: 20600000
    end_block: 20660000          # ~60k blocks so derived state commits during the run
    contracts:
      - name: LBTC
        address: "0x8236a87084f8B84306f72007F36F2618A5634494"
        events:
          - event: Transfer(address indexed from, address indexed to, uint256 value)

## 6. Define the derived state (`custom_schema.go`)

Each struct ending in **`Schema`** becomes a hot-state entity. The suffix is stripped, so
`UserPositionSchema` → `state.UserPosition` (with `.Get(key)` / `.Save(value, meta)`), and
the `// pk:` comment names the primary key.

Every event also carries **always-present metadata** — `BlockNumber`, `BlockTimestamp`,
`TransactionIndex`, `LogIndex`, ... — see [`EVENT_FIELDS.md`](../wiki/EVENT_FIELDS.md).
`Save()` automatically stamps `UpdatedAtBlock` / `UpdatedAt` from that metadata.

In [ ]:
%%writefile lbtcdemo/custom_schema.go
package lbtcdemo

import (
	"time"

	"github.com/ethereum/go-ethereum/common"
	"github.com/holiman/uint256"
)

// pk: Address
type UserPositionSchema struct {
	Address        common.Address // primary key: the account
	TotalIn        uint256.Int    // cumulative LBTC received
	TotalOut       uint256.Int    // cumulative LBTC sent
	TransferCount  uint64         // transfers touching this account
	UpdatedAtBlock uint64         // set automatically by Save()
	UpdatedAt      time.Time      // set automatically by Save()
}

## 7. Write the processor (`custom_processor.go`)

`Process(state, block)` runs once per parsed block. Iterate `block.EventsIter()` and
type-switch on the generated event structs (`Transfer` on contract `LBTC` →
`*generated.LBTCTransfer`). Register it in `init()` under `generated.ProjectName` so the
registered name always matches your config.

Note the import comments — the module-relative `generated` import and the public `sqd` facade.

`Save()` doesn't hit ClickHouse immediately — it updates an in-memory hot-state cache that the runtime commits on an interval: every `SQD_COMMIT_INTERVAL` blocks (default 5000) **or** every `SQD_COMMIT_MAX_INTERVAL` (default 3s), whichever comes first. Coming from Subsquid's `processor.run`, this is the framework-managed version of the end-of-batch `ctx.store.save([...])`: you accumulate per block, the framework flushes on the interval. Details: [`CUSTOM_PROCESSOR.md`](../wiki/CUSTOM_PROCESSOR.md).

In [ ]:
%%writefile lbtcdemo/custom_processor.go
package lbtcdemo

import (
	"github.com/ethereum/go-ethereum/common"

	// Your OWN generated package. Import path = "<module>/generated".
	// `sqd-go` writes it during --state; don't edit it by hand.
	generated "lbtcdemo/generated"

	// Public facade. Import this, never the module's internal/ packages,
	// so the project can build as its own module.
	"github.com/franz101/sqd-go/sqd"
)

func Process(state *generated.State, block *generated.ParsedBlock) error {
	var zero common.Address
	for ev := range block.EventsIter() {
		e, ok := ev.(*generated.LBTCTransfer)
		if !ok {
			continue
		}
		if e.From != zero { // debit sender
			pos, ok := state.UserPosition.Get(e.From)
			if !ok {
				pos = &generated.UserPosition{Address: e.From}
			}
			pos.TotalOut.Add(&pos.TotalOut, &e.Value)
			pos.TransferCount++
			state.UserPosition.Save(pos, e.EventMeta)
		}
		if e.To != zero { // credit receiver
			pos, ok := state.UserPosition.Get(e.To)
			if !ok {
				pos = &generated.UserPosition{Address: e.To}
			}
			pos.TotalIn.Add(&pos.TotalIn, &e.Value)
			pos.TransferCount++
			state.UserPosition.Save(pos, e.EventMeta)
		}
	}
	return nil
}

func init() {
	generated.CustomProcessFn = Process
	sqd.RegisterProcessor(generated.ProjectName, func() (sqd.Processor, error) {
		return generated.NewProcessor(sqd.GetProtoMode())
	})
}

## 8. Index with `--state`

`--state` is what makes your processor run: it regenerates the project, **compiles your
`Process` into a fresh binary**, and execs it. (Plain `start` uses the prebuilt CLI, whose
processor registry is empty — it would index raw events but skip your custom logic.)

We pass **`--no-proto`**. The single-function `Process(state, block)` API runs in the
default (V1) decode path; proto mode is an advanced performance mode that needs a separate
`ProcessProto`. If you forget `--no-proto` here, sqd-go now **fails loudly** telling you so
(rather than silently leaving the derived tables empty).

In [ ]:
!CLICKHOUSE_HOST=127.0.0.1 \
 CLICKHOUSE_NATIVE_PORT=9000 CLICKHOUSE_HTTP_PORT=8123 \
 CLICKHOUSE_USER=default CLICKHOUSE_PASSWORD='' \
 sqd-go start lbtcdemo --start-block 20600000 --end-block 20660000 --restart --parallel-fetch --state --no-proto

## 9. Query the derived state

`lbtc_transfer_events` holds the raw events; `user_positions` holds the per-address state your
processor built. Query `user_positions` with `FINAL` to collapse to the latest row per key.

In [ ]:
!./clickhouse client --query "SHOW TABLES FROM lbtcdemo"

In [ ]:
!./clickhouse client --query "SELECT count() AS raw_transfers FROM lbtcdemo.lbtc_transfer_events"
!./clickhouse client --query "SELECT count() AS positions FROM lbtcdemo.user_positions FINAL"

In [ ]:
# Top accounts by total received (FINAL collapses to the latest row per key)
!./clickhouse client --query "SELECT concat('0x', lower(hex(address))) AS account, total_in, total_out, transfer_count, updated_at_block FROM lbtcdemo.user_positions FINAL ORDER BY total_in DESC LIMIT 10 FORMAT PrettyCompact"

## 10. Metrics & tuning

While indexing, sqd-go prints a `stats` line (throughput in `blk/s`, events, checkpoint) and
a `profile` line (time in fetch / parse / insert / custom) every ~10s. Knobs like
`SQD_PARALLEL_FETCHERS`, `SQD_PARALLEL_PAGE_SIZE`, and `SQD_COLDCACHE_MB` tune throughput and
memory; CPU profiling is `--cpuprofile cpu.prof`. Full reference: [`METRICS.md`](../wiki/METRICS.md).

## Troubleshooting

| Symptom | Cause | Fix |
| --- | --- | --- |
| `found packages X and Y` | `custom_schema.go` / `custom_processor.go` use different `package` names | Use one package name |
| `use of internal package not allowed` | imported `github.com/franz101/sqd-go/internal/...` | Import `.../sqd` instead |
| `undefined: generated.SomethingTransfer` | event isn't in `config.yaml`, or wrong contract/event name | The type is `<Contract><Event>`; add the event and re-run |
| derived tables empty + a loud error about proto mode | ran `--state` without `--no-proto` | add `--no-proto` |
| derived tables empty, run looks fine, **no** error | ran plain `start` (empty registry) | add `--state` |
| start block ignored | used `--start-block=N` (the `=` form) | use space form: `--start-block N` |
| `cannot find package ...` | Import path is wrong or dependency missing | Check import path matches module name, run `go mod tidy` |

More Go-language errors and beginner debugging tips are in **Appendix — Go for beginners** below.

### Getting help

- **Modules**: [`GO_MODULES.md`](../wiki/GO_MODULES.md) — how Go modules work in sqd-go
- **Events**: [`EVENT_FIELDS.md`](../wiki/EVENT_FIELDS.md) — standard event fields
- **Metrics**: [`METRICS.md`](../wiki/METRICS.md) — performance monitoring
- **Go concepts**: [`GO_FOR_BEGINNERS.md`](../wiki/GO_FOR_BEGINNERS.md) — Go tutorial
- **Examples**: the `examples/` directory — working indexers you can study

## Appendix — Go for beginners

The tutorial above stays terse on purpose. If you're new to Go, everything below expands on the concepts it uses. None of it is required to run the notebook.

### Go concepts you'll encounter

**Packages and modules**
- **Package**: a folder of Go files sharing a `package` declaration (e.g. `package main`)
- **Module**: a collection of packages with a `go.mod` file that manages dependencies
- **Import**: how you pull in other packages: `import "fmt"` or `import "github.com/franz101/sqd-go/sqd"`

**Types and structs**
- **Struct**: groups data together: `type User struct { Name string; Age int }`
- **Pointer**: references a memory location: `var p *User = &User{Name: "Alice"}`
- **Interface**: defines behavior: `type Event interface { Meta() EventMeta }`

**Functions and methods**
- **Function**: standalone code: `func add(a, b int) int { return a + b }`
- **Method**: a function attached to a type: `func (u User) Greet() string { return "Hi " + u.Name }`
- **init()**: a special function that runs automatically at program startup

**Error handling**
- Functions return `result, error` — there are no exceptions
- Always check `if err != nil { return err }`
- `err == nil` means the operation succeeded

**Syntax sugar**
- **Short declaration**: `name := "Alice"` (type inferred)
- **Multiple return**: `return result, err`
- **Range loops**: `for i, item := range slice { ... }`
- **Type assertions**: `val, ok := event.(*Transfer)`

### Concepts in `custom_schema.go`

- **`package lbtcdemo`**: all Go files in a directory share one package name
- **`import`**: loads external packages (`time`, `common.Address`, `uint256.Int`)
- **`type ... struct`**: defines a data structure with named fields
- **`// pk: Address`**: marks which field is the primary key
- **Field types**, and why each one:
  - `common.Address` — a 20-byte Ethereum address (not a plain string)
  - `uint256.Int` — token amounts can reach `2^256 - 1`, larger than any built-in int
  - `uint64` — counts and block numbers
  - `time.Time` — Go's built-in timestamp type

### Concepts in `custom_processor.go`

- **`func Process(...) error`**: takes parameters, returns an error (`nil` = success)
- **`*generated.State`**: a pointer to State (efficient, and lets you mutate it)
- **`for ... range`**: Go's iteration loop
- **`ev.(*generated.LBTCTransfer)`**: type assertion to the concrete event type
- **`, ok` pattern**: the comma-ok idiom for a safe type assertion
- **`state.UserPosition.Get(...)`**: look up existing state
- **`&generated.UserPosition{...}`**: the address of a new struct (a pointer)
- **`pos.TotalOut.Add(...)`**: modifies a `uint256.Int` in place
- **`state.UserPosition.Save(...)`**: persist updated state to ClickHouse
- **`func init()`**: runs automatically at program startup

Key patterns:
1. **Zero-address check**: `var zero common.Address` is the all-zero address
2. **Type assertion**: `e, ok := ev.(*generated.LBTCTransfer)` checks the event type
3. **Get-or-create**: `pos, ok := state.UserPosition.Get(key); if !ok { pos = &... }`
4. **In-place mutation**: `pos.TotalOut.Add(&pos.TotalOut, &e.Value)`
5. **State persistence**: `state.UserPosition.Save(pos, e.EventMeta)`

`e.EventMeta` carries the block number, timestamp, and transaction hash — the fields ClickHouse needs for ordering and time-travel queries.

### More Go-language errors

| Error message | What it means | How to fix |
| --- | --- | --- |
| `found packages X and Y in directory` | different package names in one directory | make both files use `package lbtcdemo` |
| `use of internal package ... not allowed` | importing `internal/` from another module | import `github.com/franz101/sqd-go/sqd` instead |
| `undefined: generated.LBTCTransfer` | type doesn't exist in generated code | check the event name in `config.yaml` (case-sensitive!) |
| `cannot use ... as type ...` | type mismatch | check the types match (e.g. `common.Address` vs `string`) |
| `expected ..., found ...` | wrong number or type of arguments | match the function signature |
| `syntax error: unexpected ...` | missing brace or semicolon | usually a missing `}` — check the line above |

### Debugging tips

1. Read the error message — Go errors include `file:line`
2. `go fmt custom_processor.go` — auto-format
3. `go vet ./...` — find additional issues
4. Make sure every imported package is actually used
5. Match package names across all files in a directory
6. Check pointer vs value: does the function want `*Type` or `Type`?
7. Only upper-case names (e.g. `Process`) are visible outside a package